# 1. CLONE REPO

In [6]:
%cd /kaggle/working
!git clone https://github.com/PhuongThao-2005/LViT.git

%cd /kaggle/working/LViT
!git checkout BTRXD-LViT-T-SlidingTrain

/kaggle/working
fatal: destination path 'LViT' already exists and is not an empty directory.
/kaggle/working/LViT
M	Config.py
Already on 'BTRXD-LViT-T-SlidingTrain'
Your branch is up to date with 'origin/BTRXD-LViT-T-SlidingTrain'.


# 2. COPY DATASET FULLSIZE

In [7]:
!cp -r /kaggle/input/datasets/phuongthao205/btrxd-fullsize/BTRXD_fullsize /kaggle/working/LViT/datasets/

# 3. INSTALL PACKAGES

In [8]:
!pip install -q openpyxl tensorboardX thop transformers ml-collections scipy scikit-learn

# 4. CONFIG

In [9]:
%%writefile /kaggle/working/LViT/Config.py

# -*- coding: utf-8 -*-

import os
import time
import random
import numpy as np
import torch
import ml_collections

# BASIC
save_model = True
tensorboard = True
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
use_cuda = torch.cuda.is_available()
seed = 666
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# TRAIN CONFIG
cosineLR = True
n_channels = 3
n_labels = 1
epochs = 200
img_size = 224
print_frequency = 50
save_frequency = 10
vis_frequency = 50
early_stopping_patience = 50
pretrain = False

task_name = 'BTRXD_fullsize'

learning_rate = 3e-4
batch_size = 16
accumulation_steps = 1

model_name = 'LViT'

# DATASET
train_dataset = './datasets/' + task_name + '/Train_Folder/'
val_dataset   = './datasets/' + task_name + '/Val_Folder/'
test_dataset  = './datasets/' + task_name + '/Test_Folder/'
task_dataset  = './datasets/' + task_name + '/Train_Folder/'
label_plan_csv = './datasets/' + task_name + '/label_plan_100.csv'

# SESSION
session_name = 'PATCH_LViT_' + time.strftime('%m.%d_%Hh%M')
base_save_dir = '/kaggle/working/'
save_path          = os.path.join(base_save_dir, task_name, model_name, session_name) + os.sep
model_path         = os.path.join(save_path, 'models') + os.sep
tensorboard_folder = os.path.join(save_path, 'tensorboard_logs') + os.sep
logger_path        = os.path.join(save_path, session_name + '.log')
visualize_path     = os.path.join(save_path, 'visualize_val') + os.sep

# MODEL CONFIG
def get_CTranS_config():
    config = ml_collections.ConfigDict()
    config.transformer = ml_collections.ConfigDict()
    config.KV_size = 960
    config.transformer.num_heads = 4
    config.transformer.num_layers = 4
    config.expand_ratio = 4
    config.transformer.embeddings_dropout_rate = 0.1
    config.transformer.attention_dropout_rate = 0.1
    config.transformer.dropout_rate = 0
    config.patch_sizes = [16, 8, 4, 2]
    config.base_channel = 64
    config.n_classes = 1
    return config

Overwriting /kaggle/working/LViT/Config.py


# 5. CHECK DATASET & PATCH COUNT

In [10]:
import sys
sys.path.insert(0, '/kaggle/working/LViT')

import os
import cv2
import numpy as np

os.chdir('/kaggle/working/LViT')

from patch_dataset import PatchDataset
from utils import read_text

train_text = read_text(
    './datasets/BTRXD_fullsize/Train_Folder/Train_text.xlsx'
)

ds = PatchDataset(
    './datasets/BTRXD_fullsize/Train_Folder',
    train_text,
    augment=False,
    seed=666
)

print(f"Train patches: {len(ds)}")
sample = ds[0]

print(f"image: {sample['image'].shape}")
print(f"label: {sample['label'].shape}")
print(f"text : {sample['text'].shape}")

# PATCH DISTRIBUTION

pos_count = 0
for p in ds.patches:
    mask = cv2.imread(p[1], 0)
    y0 = p[2]
    x0 = p[3]
    patch = mask[y0:y0+224, x0:x0+224]
    if patch.max() > 0:
        pos_count += 1

neg_count = len(ds) - pos_count

print("\nPatch Distribution")
print("----------------------------")
print(f"Positive patches : {pos_count}")
print(f"Negative patches : {neg_count}")
print(f"Pos:Neg ratio    : 1:{neg_count/max(pos_count,1):.2f}")

/kaggle/working/LViT/utils.py:536: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df.Description[i] = df.Description[i] + ' EOF XXX' * (9 - count)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PatchDataset ready: 10980 patches | neg_ratio=1 | seed=666
Train patches: 10980
image: torch.Size([3, 224, 224])
label: torch.Size([1, 224, 224])
text : torch.Size([10, 768])

Patch Distribution
----------------------------
Positive patches : 5733
Negative patches : 5247
Pos:Neg ratio    : 1:0.92


# 6. TRAIN

In [11]:
!python train_patch.py

Traceback (most recent call last):
  File "/kaggle/working/LViT/train_patch.py", line 15, in <module>
    from LV_loss.loss import BinaryDiceLoss
ModuleNotFoundError: No module named 'LV_loss.loss'


# 7. INFERENCE SLIDING WINDOW

In [12]:
# FIND PATCH TRAINING CHECKPOINT
import glob
import os

ckpts = sorted(
    glob.glob('/kaggle/working/BTRXD_fullsize/LViT/PATCH_LViT_*/models/best_model.pth.tar')
)

print("Checkpoints found:", len(ckpts))
if len(ckpts) > 0:
    print("\nAll checkpoints:")
    for c in ckpts:
        print(" ", c)

    latest_ckpt = ckpts[-1]
    SESSION_2B = latest_ckpt.split('/')[-4]

    print(f"\nUse the latest checkpoint: {latest_ckpt}")
    print(f"Session: {SESSION_2B}")
    
else:
    print("\nNo checkpoint found.")
    print("Training probably crashed before saving best_model.pth.tar")

FileNotFoundError: Checkpoint PATCH_LViT_* NOT FOUND!

In [ ]:
# Set test session

config_path = '/kaggle/working/LViT/Config.py'
with open(config_path, 'r') as f:
    lines = f.readlines()

# remove old test_session
lines = [l for l in lines if not l.strip().startswith('test_session')]

# append new
lines.append(f'\ntest_session = "{SESSION_2B}"\n')
with open(config_path, 'w') as f:
    f.writelines(lines)

print("Updated Config.py")
!grep "test_session" /kaggle/working/LViT/Config.py

# Run full image inference

!python -u test_sliding_window.py

# 8. RESULT

In [ ]:
import csv

result_path = f'/kaggle/working/BTRXD_fullsize/LViT/{SESSION_2B}/sliding_window_test/results.txt'
print("=" * 55)
print("Phase 2B Sliding Window Results")
print("=" * 55)
print(open(result_path).read())

# Compare 3 phases
print("=" * 55)
print("Compare all phases")
print("=" * 55)
print(f"  Phase 1  (resize train+infer)     : Dice = 0.6856")
print(f"  Phase 2A (resize train, SW infer) : Dice = 0.0116")
print(f"  Phase 2B (patch train, SW infer)  : Dice = ???")

# Training log
log_path = f'/kaggle/working/BTRXD_fullsize/LViT/{SESSION_2B}/training_log.csv'
if os.path.exists(log_path):
    with open(log_path) as f:
        rows = list(csv.DictReader(f))
    print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Val Dice':>9} | {'Pred Ratio':>10} | Val Set")
    print("-" * 60)
    for r in rows:
        print(f"{r['epoch']:>6} | {float(r['train_loss']):>10.4f} | "
              f"{float(r['val_dice']):>9.4f} | {float(r['pred_ratio']):>10.5f} | {r['val_subset']}")

# 9. VISUALIZATION

In [ ]:
# Visualization: Image | GT | Prediction
import cv2, numpy as np, matplotlib.pyplot as plt, os, glob, torch
from test_sliding_window import sliding_window_predict
from Load_Dataset import HFTextEmbedder
from utils import read_text
import Config as config
from nets.LViT import LViT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model Phase 2B
cfg_vit = config.get_CTranS_config()
model   = LViT(cfg_vit, n_channels=config.n_channels, n_classes=config.n_labels)
ckpt    = torch.load(latest_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
model   = model.to(device).eval()
val_dice = ckpt.get('val_dice', -1)
print(
    f"Loaded: {latest_ckpt} "
    f"(epoch={ckpt.get('epoch', '?')}, "
    f"val_dice={val_dice:.4f})"
)

# Load text + embedder
test_text = read_text(os.path.join(config.test_dataset, 'Test_text.xlsx'))
embedder  = HFTextEmbedder(model_name="bert-base-uncased", max_tokens=10)

# Lấy 5 ảnh test để visualize
img_dir  = os.path.join(config.test_dataset, 'img')
mask_dir = os.path.join(config.test_dataset, 'labelcol')
samples  = sorted(os.listdir(img_dir))[:5]

fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4 * len(samples)))
fig.suptitle(f'Phase 2B Predictions — {SESSION_2B}', fontsize=13, fontweight='bold')

for row, img_fn in enumerate(samples):
    stem = os.path.splitext(img_fn)[0]
    image_bgr = cv2.imread(os.path.join(img_dir, img_fn))
    H, W = image_bgr.shape[:2]

    # Load GT mask
    mask_fn = os.path.join(mask_dir, img_fn)
    if not os.path.exists(mask_fn):
        mask_fn = os.path.join(mask_dir, stem + '.png')
    gt_mask = cv2.imread(mask_fn, 0)
    gt_mask = cv2.resize(gt_mask, (W, H), interpolation=cv2.INTER_NEAREST)
    gt_bin  = (gt_mask > 0).astype(np.uint8)

    # Sliding window predict
    mask_basename = os.path.basename(mask_fn)
    text_str = test_text.get(mask_basename, test_text.get(img_fn, 'tumor lesion segmentation'))
    text_emb = embedder.encode(text_str)
    text_t   = torch.from_numpy(text_emb).unsqueeze(0).float()

    with torch.no_grad():
        prob_map = sliding_window_predict(model, image_bgr, text_t, device=device)
    pred_bin = (prob_map > 0.5).astype(np.uint8)

    # Tính Dice cho ảnh này
    p = pred_bin.reshape(-1).astype(np.float32)
    g = gt_bin.reshape(-1).astype(np.float32)
    dice = float(2 * np.sum(p * g) / (np.sum(p) + np.sum(g) + 1e-5))

    # Plot
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    axes[row, 0].imshow(image_rgb); axes[row, 0].set_title(f'{img_fn}\n({W}×{H})')
    axes[row, 1].imshow(gt_bin, cmap='gray', vmin=0, vmax=1); axes[row, 1].set_title('Ground Truth')
    axes[row, 2].imshow(pred_bin, cmap='gray', vmin=0, vmax=1)
    axes[row, 2].set_title(f'Prediction\nDice={dice:.4f}')

    for ax in axes[row]: ax.axis('off')

plt.tight_layout()
out_vis = f'/kaggle/working/BTRXD_fullsize/LViT/{SESSION_2B}/sliding_window_test/visualization.png'
plt.savefig(out_vis, dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved visualization → {out_vis}")